## Data paths

Private Colab and Google Drive paths were replaced with repository-relative `data/` paths. Place permitted local inputs under `data/` or adapt the path variables for your environment. Original private research data are not included.

# Paper 2 — Fixed OpenAI Textual-Pattern Analysis for the Primary Assignment Corpus

This notebook repeats and fixes the parts that previously failed. It is designed specifically for the **primary Paper 2 assignment corpus** of 543 files, not the controlled testing sample.

## What this fixed version changes

1. **Uses the primary Paper 2 assignment-level file list** (`paper2_assignment_level_high_impact_conflict_flags.csv` or the combined detector results sheet).
2. **Rejects controlled-testing-sample files** and does not read `Conflict_Analysis_Outputs/openai_qualitative_reviews.csv`.
3. **Fixes the year-grouping issue** so `Medical` is not treated as a year category.
4. **Resolves text paths only from Primary Sample / Trimmed Assignments**.
5. **Runs OpenAI feature coding on de-identified trimmed assignment excerpts** when `RUN_OPENAI = True`.
6. Exports completed corpus-level outputs: feature codes, feature prevalence, APA-ready tables, charts, manual-validation sample, and manuscript-ready interpretation.

The OpenAI feature coding is exploratory and corpus-level. It must **not** be used to claim that any individual assignment file was AI-generated.

In [ ]:
import os
import re
import sys
import json
import math
import time
import random
import warnings
import subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 180)

try:
    from statsmodels.stats.multitest import multipletests

    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

try:

    DOCX_AVAILABLE = True
except Exception:
    DOCX_AVAILABLE = False

INSTALL_OPENAI_PACKAGE = False
if INSTALL_OPENAI_PACKAGE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

DEFAULT_DATA_ROOT = "data" if IN_COLAB else "data"
DATA_ROOT = Path(os.environ.get("RESEARCH_DATA_ROOT", DEFAULT_DATA_ROOT))
MYDRIVE = DATA_ROOT

RUN_OPENAI = True
OPENAI_MODEL = "gpt-4o-mini"

EXPECTED_N_FILES = 543
EXPECTED_HIGH_CONFLICT_N = 78
MATCH_RATIO = 1
MATCH_VARS = ["language", "assignment_type"]
RANDOM_SEED = 20260615
CHUNK_WORDS = 700
MAX_OPENAI_FILES = None
INCLUDE_CODE_FILES_FOR_OPENAI = False

ASSIGNMENT_RESULTS_PATH_OVERRIDE = None
TRIMMED_TEXT_ROOT_OVERRIDE = None

DEFAULT_OUTPUT_DIR = Path("outputs") / "openai_textual_feature_coding"
OUTPUT_DIR = Path(os.environ.get("RESEARCH_OUTPUT_DIR", DEFAULT_OUTPUT_DIR))
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
CACHE_DIR = OUTPUT_DIR / "cache"
for path in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("IN_COLAB:", IN_COLAB)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN_OPENAI:", RUN_OPENAI)

## 1. Load the primary Paper 2 assignment-level detector results

This section deliberately prefers `paper2_assignment_level_high_impact_conflict_flags.csv`, which belongs to the **primary 543-file assignment corpus**. It rejects files from `Conflict_Analysis_Outputs` because those belong to the controlled testing sample, not the Paper 2 assignment corpus.

In [ ]:
import os
from pathlib import Path

ASSIGNMENT_RESULTS_PATH_OVERRIDE = os.getenv("ASSIGNMENT_RESULTS_PATH_OVERRIDE")
RESULTS_SEARCH_ROOTS = [
    Path(root)
    for root in os.getenv("ASSIGNMENT_RESULTS_SEARCH_ROOTS", "").split(os.pathsep)
    if root
] or [Path.cwd(), Path("/mnt/data")]


def safe_exists(path):
    try:
        return Path(path).exists()
    except OSError:
        return False


def find_candidate_files():
    candidates = []

    if ASSIGNMENT_RESULTS_PATH_OVERRIDE and safe_exists(
        ASSIGNMENT_RESULTS_PATH_OVERRIDE
    ):
        candidates.append(Path(ASSIGNMENT_RESULTS_PATH_OVERRIDE))

    filename_patterns = [
        "paper2_assignment_level_high_impact_conflict_flags.csv",
        "AI_Detection_Results_Combined.xlsx",
        "AI_Detection_Results_Combined.csv",
    ]

    for root in RESULTS_SEARCH_ROOTS:
        if not root.exists():
            continue
        for pattern in filename_patterns:
            try:
                candidates.extend(root.rglob(pattern))
            except OSError:
                continue

    unique = []
    seen = set()
    for candidate in candidates:
        try:
            key = str(candidate.resolve())
        except OSError:
            key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique.append(candidate)
    return unique


candidates = find_candidate_files()
print("Candidate assignment-level files found:")
for index, candidate in enumerate(candidates[:30], 1):
    print(f"{index:02d}. {candidate}")

# Exclude controlled testing-sample outputs from the primary analysis.
primary_candidates = [
    candidate
    for candidate in candidates
    if "Conflict_Analysis_Outputs" not in str(candidate)
    and "Testing Sample" not in str(candidate)
]

if not primary_candidates:
    raise FileNotFoundError(
        "No primary assignment-level results file found. Set "
        "ASSIGNMENT_RESULTS_PATH_OVERRIDE or ASSIGNMENT_RESULTS_SEARCH_ROOTS."
    )

primary_candidates.sort(
    key=lambda path: (
        "paper2_assignment_level_high_impact_conflict_flags" not in path.name,
        len(str(path)),
    )
)
DATA_PATH = primary_candidates[0]
print("\nUsing primary assignment results file:", DATA_PATH)

if DATA_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_raw = pd.read_excel(DATA_PATH)
else:
    df_raw = pd.read_csv(DATA_PATH)

print("Loaded shape:", df_raw.shape)
display(df_raw.head())

## 2. Standardize columns, fix year grouping, and validate the 543-file corpus

This section fixes the previous `Medical` year-grouping problem. `Medical` is a major/path segment, not a year. Medical OSCE and Manuscript files are mapped to First Year; Medical Research Project files are mapped to Second Year. KFUPM semester-specific first-year rows are collapsed to First Year for the paper-level year grouping.

In [ ]:
import re

import numpy as np
import pandas as pd

# Set to an integer to validate the expected corpus size; otherwise skip this check.
EXPECTED_N_FILES = None


def normalize_colname(column):
    return re.sub(r"[^a-z0-9]+", "_", str(column).strip().lower()).strip("_")


rename_map = {column: normalize_colname(column) for column in df_raw.columns}
df = df_raw.rename(columns=rename_map).copy()
print("Standardized columns:")
print(list(df.columns))

ALIASES = {
    "file_name": [
        "file_name",
        "filename",
        "file",
        "document",
        "document_name",
        "name",
        "source_file",
        "txt_file",
        "pdf_file",
    ],
    "language": ["language", "submission_language", "lang", "language_norm"],
    "university": ["university", "institution", "school", "university_group"],
    "major": ["major", "course_major", "discipline", "field", "major_group"],
    "assignment_type": [
        "assignment_type",
        "assignment",
        "type",
        "task_type",
        "assignment_context",
    ],
    "year_grouping": ["year_grouping", "year", "academic_year", "year_group", "level"],
    "word_count": [
        "word_count",
        "words",
        "n_words",
        "word_count_reported",
        "clean_word_count",
        "post_cleaning_word_count",
    ],
    "combined_ai_rate_100": [
        "combined_ai_rate_100",
        "combined_ai_rate_percent",
        "combined_ai_rate_pct",
        "combined_ai_rate",
        "combined_ai",
        "combined_score",
        "combined_ai_rate_percent_",
        "combined_weighted_ai_rate",
        "weighted_combined_ai_rate",
        "weighted_combined_ai_rate_percent",
        "weighted_ai_rate",
        "weighted_ai_rate_percent",
        "weighted_ai_score",
        "final_combined_ai_rate",
        "final_ai_rate",
        "ai_rate_combined",
        "combined_rate",
    ],
    "gptzero_ai_rate_percent": [
        "gptzero_ai_rate_percent",
        "gptzero_score_percent",
        "gptzero_ai_rate",
        "gptzero",
        "gptzero_percent",
        "gptzero_rate",
    ],
    "pangram_ai_rate_percent": [
        "pangram_ai_rate_percent",
        "pangram_score_percent",
        "pangram_ai_rate",
        "pangram",
        "pangram_percent",
        "pangram_rate",
    ],
    "sapling_ai_rate_percent": [
        "sapling_ai_rate_percent",
        "sapling_score_percent",
        "sapling_ai_rate",
        "sapling",
        "sapling_percent",
        "sapling_rate",
    ],
    "isgen_ai_rate_percent": [
        "isgen_ai_rate_percent",
        "isgen_score_percent",
        "isgen_ai_rate",
        "isgen",
        "isgen_percent",
        "isgen_rate",
    ],
}

# Override automatic detection only when a source column is ambiguous.
MANUAL_COLUMNS = {}

resolved = {}
for canonical, aliases in ALIASES.items():
    if canonical in MANUAL_COLUMNS:
        column = normalize_colname(MANUAL_COLUMNS[canonical])
        if column not in df.columns:
            raise ValueError(
                f"Manual column for {canonical} was set to {column}, but it is not in the loaded data."
            )
        resolved[canonical] = column
        continue

    for alias in aliases:
        column = normalize_colname(alias)
        if column in df.columns:
            resolved[canonical] = column
            break

print("Resolved column map:")
for canonical, column in resolved.items():
    print(f"  {canonical}: {column}")

required_base = [
    "file_name",
    "language",
    "university",
    "major",
    "assignment_type",
    "year_grouping",
    "word_count",
    "pangram_ai_rate_percent",
]
missing_base = [column for column in required_base if column not in resolved]
if missing_base:
    print("\nAvailable standardized columns:")
    print(list(df.columns))
    raise ValueError(
        f"Missing required columns after alias detection: {missing_base}. "
        "Update ALIASES or MANUAL_COLUMNS."
    )

# Rename detected fields to the analysis schema.
df = df.rename(columns={source: canonical for canonical, source in resolved.items()})

for column in [
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]:
    if column not in df.columns:
        df[column] = np.nan

for column in [
    "word_count",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Use the study's language-specific calibration weights when no combined score is supplied.
LANGUAGE_SPECIFIC_WEIGHTS = {
    "arabic": {"gptzero": 0.24, "pangram": 0.33, "sapling": 0.16, "isgen": 0.27},
    "english": {"gptzero": 0.27, "pangram": 0.28, "sapling": 0.24, "isgen": 0.21},
}


def normalize_language_for_weights(value):
    language = str(value).strip().lower()
    if language in {"arabic", "arab", "ar", "عربي", "arabic text"}:
        return "arabic"
    if language in {"english", "eng", "en", "english text"}:
        return "english"
    if language in {"code", "coding", "programming"}:
        return "code"
    return language


def is_coding_assignment_value(value):
    assignment = str(value).strip().lower()
    return any(term in assignment for term in ("code", "coding", "program"))


def weighted_combined_ai_rate_fallback(row):
    language = normalize_language_for_weights(row.get("language", ""))
    if language == "code" or is_coding_assignment_value(row.get("assignment_type", "")):
        return row.get("pangram_ai_rate_percent", np.nan)

    weights = LANGUAGE_SPECIFIC_WEIGHTS.get(
        language, LANGUAGE_SPECIFIC_WEIGHTS["english"]
    )
    values = {
        detector: float(row[column])
        for detector, column in {
            "gptzero": "gptzero_ai_rate_percent",
            "pangram": "pangram_ai_rate_percent",
            "sapling": "sapling_ai_rate_percent",
            "isgen": "isgen_ai_rate_percent",
        }.items()
        if pd.notna(row.get(column, np.nan))
    }

    if not values:
        return np.nan

    weight_sum = sum(weights[detector] for detector in values)
    if weight_sum == 0:
        return np.nan

    return sum(values[detector] * weights[detector] for detector in values) / weight_sum


if "combined_ai_rate_100" in df.columns:
    df["combined_ai_rate_100"] = pd.to_numeric(
        df["combined_ai_rate_100"], errors="coerce"
    )
    nonmissing = df["combined_ai_rate_100"].dropna()
    if len(nonmissing) and nonmissing.max() <= 1.000001:
        df["combined_ai_rate_100"] *= 100
        print("Detected Combined AI rate on a 0-1 scale; converted to 0-100.")
    print("Using precomputed Combined AI rate column.")
else:
    df["combined_ai_rate_100"] = df.apply(weighted_combined_ai_rate_fallback, axis=1)
    print(
        "Derived combined_ai_rate_100 from detector rates using language-specific weights."
    )

if df["combined_ai_rate_100"].notna().sum() == 0:
    print("\nAvailable standardized columns:")
    print(list(df.columns))
    raise ValueError(
        "combined_ai_rate_100 could not be created. Check detector score columns "
        "or set MANUAL_COLUMNS."
    )

for column in [
    "language",
    "university",
    "major",
    "assignment_type",
    "year_grouping",
    "file_name",
]:
    df[column] = df[column].astype(str).str.strip()
    df.loc[df[column].isin(["nan", "None", "", "<NA>"]), column] = pd.NA


def fix_year_grouping(row):
    year = str(row.get("year_grouping", "")).strip()
    major = str(row.get("major", "")).strip()
    assignment_type = str(row.get("assignment_type", "")).strip()

    # Reclassify records where a major was incorrectly stored as the year grouping.
    if year.lower() == "medical" and major.lower() in {"med", "medical", "medicine"}:
        if assignment_type in {"OSCE", "Manuscripts"}:
            return "First Year"
        if assignment_type in {"Research Projects", "Research Project"}:
            return "Second Year"
        return pd.NA

    if "First Year" in year:
        return "First Year"
    if "Second Year" in year:
        return "Second Year"
    return year or pd.NA


df["year_grouping_original"] = df["year_grouping"].copy()
df["year_grouping"] = df.apply(fix_year_grouping, axis=1)

print("Corpus n:", len(df))
print("Combined AI rate non-missing:", int(df["combined_ai_rate_100"].notna().sum()))
print("High-level year grouping counts after fix:")
print(df["year_grouping"].value_counts(dropna=False).to_frame("n"))

if EXPECTED_N_FILES is not None and len(df) != EXPECTED_N_FILES:
    print(f"WARNING: Expected {EXPECTED_N_FILES} files, but loaded {len(df)}.")

if df["year_grouping"].astype(str).str.lower().eq("medical").any():
    raise ValueError("Year grouping still contains 'Medical'.")

## 3. Reproduce the high-impact detector-conflict flag

Definition: at least one detector score ≤ 30%, at least one detector score ≥ 70%, and Combined AI rate between 40% and 60%.

In [ ]:
# =========================================================
# 3. Detector metrics and high-impact conflict flag
# =========================================================
score_cols = [
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]
score_matrix = df[score_cols].apply(pd.to_numeric, errors="coerce")

df["combined_ai_rate_100"] = pd.to_numeric(df["combined_ai_rate_100"], errors="coerce")
df["combined_ai_rate_01"] = df["combined_ai_rate_100"] / 100.0

df["detector_score_min"] = score_matrix.min(axis=1, skipna=True)
df["detector_score_max"] = score_matrix.max(axis=1, skipna=True)
df["detector_score_range"] = df["detector_score_max"] - df["detector_score_min"]
df["detector_score_variance"] = score_matrix.var(axis=1, skipna=True)
df["n_detector_scores_available"] = score_matrix.notna().sum(axis=1)
df["any_detector_le_30"] = score_matrix.le(30).any(axis=1)
df["any_detector_ge_70"] = score_matrix.ge(70).any(axis=1)
df["combined_mid_40_60"] = df["combined_ai_rate_100"].between(40, 60, inclusive="both")
df["high_impact_conflict"] = (
    df["any_detector_le_30"] & df["any_detector_ge_70"] & df["combined_mid_40_60"]
)

for tool_col in score_cols:
    prefix = tool_col.replace("_ai_rate_percent", "")
    df[f"{prefix}_le_30"] = df[tool_col].le(30)
    df[f"{prefix}_ge_70"] = df[tool_col].ge(70)

n_high = int(df["high_impact_conflict"].sum())
print(f"High-impact conflict files: {n_high} / {len(df)} ({100*n_high/len(df):.1f}%)")
if n_high != EXPECTED_HIGH_CONFLICT_N:
    raise ValueError(
        f"Expected {EXPECTED_HIGH_CONFLICT_N} high-impact conflict files but found {n_high}. Check the input dataset and detector columns."
    )

# Save a fixed flags file for this run.
flags_path = (
    OUTPUT_DIR / "paper2_assignment_level_high_impact_conflict_flags_YEAR_FIXED.csv"
)
df.to_csv(flags_path, index=False, encoding="utf-8-sig")
print("Saved fixed flags:", flags_path)

## 4. Resolve de-identified trimmed text paths from the primary sample only

This is the main fix. The notebook only accepts text files under a path containing both `Primary Sample` and `Trimmed Assignments`. It rejects `Testing Sample` paths so the OpenAI coding cannot accidentally analyze the controlled testing sample.

In [ ]:
PRIMARY_TRIMMED_TEXT_ROOTS = globals().get("PRIMARY_TRIMMED_TEXT_ROOTS", [])
TRIMMED_TEXT_ROOT_OVERRIDE = globals().get("TRIMMED_TEXT_ROOT_OVERRIDE")

TEXT_ROOT_CANDIDATES = []
if TRIMMED_TEXT_ROOT_OVERRIDE:
    TEXT_ROOT_CANDIDATES.append(Path(TRIMMED_TEXT_ROOT_OVERRIDE))
TEXT_ROOT_CANDIDATES.extend(Path(root) for root in PRIMARY_TRIMMED_TEXT_ROOTS)

existing_text_roots = [path for path in TEXT_ROOT_CANDIDATES if path.exists()]
print("Existing text roots:")
for path in existing_text_roots:
    print(" -", path)

path_candidates = [
    column
    for column in df.columns
    if any(
        token in column.lower()
        for token in ["path", "filepath", "file_path", "resolved"]
    )
]
print("Potential path columns:", path_candidates)

path_index = defaultdict(list)
for root in existing_text_roots:
    for extension in ("*.txt", "*.md"):
        for path in root.rglob(extension):
            path_string = str(path)
            if "Testing Sample" in path_string:
                continue
            if (
                "Primary Sample" not in path_string
                or "Trimmed Assignments" not in path_string
            ):
                continue
            path_index[path.name].append(path)

print(
    f"Indexed {sum(len(paths) for paths in path_index.values())} primary-sample "
    f"trimmed text files across {len(path_index)} unique filenames."
)


def clean_token(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()


def path_score_for_row(path, row):
    path_string = str(path).lower()
    score = 0

    for column, weight in [
        ("university", 4),
        ("major", 2),
        ("assignment_type", 5),
        ("year_grouping", 2),
        ("language", 1),
    ]:
        value = row.get(column)
        if pd.notna(value):
            tokens = clean_token(value).split()
            abbreviations = {
                "med": ["medical", "medicine"],
                "swe": ["software", "computer"],
                "mecheng": ["mechanical", "engineering"],
            }
            tokens.extend(abbreviations.get(str(value).lower(), []))
            if any(token and token in path_string for token in tokens):
                score += weight

    assignment_type = str(row.get("assignment_type", "")).lower()
    assignment_synonyms = {
        "osce": ["osce"],
        "manuscripts": ["manuscript", "manuscripts"],
        "research projects": ["research"],
        "article": ["article", "articles"],
        "computer project report": ["computer", "project report"],
        "coding project": ["coding", "code"],
        "laboratory report": ["lab", "laboratory"],
        "graduation projects": ["graduation"],
        "project report": ["project report"],
    }
    for label, synonyms in assignment_synonyms.items():
        if label in assignment_type and any(
            synonym in path_string for synonym in synonyms
        ):
            score += 8

    return score


def validate_primary_text_path(path):
    if path is None or pd.isna(path):
        return False

    path_string = str(path)
    return (
        "Testing Sample" not in path_string
        and "Primary Sample" in path_string
        and "Trimmed Assignments" in path_string
        and Path(path_string).exists()
    )


def resolve_text_path(row):
    for column in path_candidates:
        value = row.get(column)
        if pd.notna(value) and validate_primary_text_path(value):
            return str(Path(value))

    filename = row.get("file_name")
    if pd.isna(filename):
        return pd.NA

    matches = path_index.get(str(filename).strip(), [])
    if len(matches) == 1:
        return str(matches[0])
    if len(matches) > 1:
        # Use corpus metadata to select among duplicate filenames.
        scored_matches = sorted(
            ((path_score_for_row(path, row), path) for path in matches),
            reverse=True,
            key=lambda item: item[0],
        )
        return str(scored_matches[0][1])

    return pd.NA


df["text_path"] = df.apply(resolve_text_path, axis=1)
df["text_path_exists"] = df["text_path"].apply(
    lambda path: isinstance(path, str) and Path(path).exists()
)
df["text_path_is_primary_trimmed"] = df["text_path"].apply(
    lambda path: (
        isinstance(path, str)
        and "Primary Sample" in path
        and "Trimmed Assignments" in path
        and "Testing Sample" not in path
    )
)

print(
    "Resolved primary trimmed text paths:",
    int(df["text_path_exists"].sum()),
    "of",
    len(df),
)
missing_paths = df[~df["text_path_exists"]].copy()
missing_path_out = OUTPUT_DIR / "missing_or_unresolved_primary_trimmed_text_paths.csv"
missing_paths.to_csv(missing_path_out, index=False, encoding="utf-8-sig")
print("Missing/unresolved paths saved:", missing_path_out)

if (
    df["text_path"]
    .astype(str)
    .str.contains("Testing Sample", case=False, na=False)
    .any()
):
    raise ValueError(
        "A Testing Sample path was resolved. Stop: this notebook must analyze only the primary Paper 2 assignment corpus."
    )

if int(df["text_path_exists"].sum()) < len(df):
    print(
        "WARNING: Some primary text paths are unresolved. OpenAI coding will use only rows with resolved paths."
    )
    display(
        missing_paths[
            [
                "file_name",
                "language",
                "assignment_type",
                "major",
                "university",
                "year_grouping",
            ]
        ].head(20)
    )

## 5. Build high-conflict and matched non-high-conflict groups

All 78 high-impact conflict files are selected. The comparison group is matched first by language and assignment type, then uses language-only fallback if an exact stratum is short.

In [ ]:
def is_code_row(row):
    values = " ".join(
        str(row.get(column, ""))
        for column in ["language", "assignment_type", "major", "file_name"]
    )
    return bool(
        re.search(
            r"\b(code|coding|program|python|java|software)\b",
            values,
            flags=re.IGNORECASE,
        )
    )


df["is_code_like"] = df.apply(is_code_row, axis=1)

# Restrict coding to resolved primary text files; code-like files are optional.
eligible = df[df["text_path_exists"] & df["text_path_is_primary_trimmed"]].copy()
if not INCLUDE_CODE_FILES_FOR_OPENAI:
    eligible = eligible[~eligible["is_code_like"]].copy()

high_df = eligible[eligible["high_impact_conflict"]].copy()
non_df = eligible[~eligible["high_impact_conflict"]].copy()

matched_parts = []
match_notes = []
used = set()

for key, hgrp in high_df.groupby(MATCH_VARS, dropna=False):
    if not isinstance(key, tuple):
        key = (key,)

    target_n = len(hgrp) * MATCH_RATIO

    mask_exact = pd.Series(True, index=non_df.index)
    for column, value in zip(MATCH_VARS, key):
        if pd.isna(value):
            mask_exact &= non_df[column].isna()
        else:
            mask_exact &= non_df[column].astype(str).eq(str(value))

    selected = []
    pool_exact = non_df[mask_exact & ~non_df.index.isin(used)]
    take_exact = min(target_n, len(pool_exact))
    if take_exact > 0:
        sample = pool_exact.sample(n=take_exact, random_state=RANDOM_SEED)
        selected.append(sample)
        used.update(sample.index.tolist())

    # Preserve the matching hierarchy: exact stratum, language, then overall.
    shortfall = target_n - take_exact
    take_fallback = 0
    if shortfall > 0 and key:
        language_value = key[0]
        if pd.notna(language_value):
            mask_language = non_df["language"].astype(str).eq(str(language_value))
        else:
            mask_language = non_df["language"].isna()

        pool_language = non_df[mask_language & ~non_df.index.isin(used)]
        take_fallback = min(shortfall, len(pool_language))
        if take_fallback > 0:
            sample = pool_language.sample(
                n=take_fallback,
                random_state=RANDOM_SEED + 1,
            )
            selected.append(sample)
            used.update(sample.index.tolist())

    remaining_shortfall = target_n - take_exact - take_fallback
    take_overall = 0
    if remaining_shortfall > 0:
        pool_all = non_df[~non_df.index.isin(used)]
        take_overall = min(remaining_shortfall, len(pool_all))
        if take_overall > 0:
            sample = pool_all.sample(
                n=take_overall,
                random_state=RANDOM_SEED + 2,
            )
            selected.append(sample)
            used.update(sample.index.tolist())

    if selected:
        matched_parts.append(pd.concat(selected, axis=0))

    match_notes.append(
        {
            "match_language": key[0] if len(key) > 0 else None,
            "match_assignment_type": key[1] if len(key) > 1 else None,
            "high_conflict_n": len(hgrp),
            "target_non_conflict_n": target_n,
            "selected_exact_n": take_exact,
            "selected_language_fallback_n": take_fallback,
            "selected_overall_fallback_n": take_overall,
            "shortfall_n": max(0, remaining_shortfall - take_overall),
        }
    )

matched_non_df = (
    pd.concat(matched_parts, axis=0).copy()
    if matched_parts
    else pd.DataFrame(columns=eligible.columns)
)

high_df["conflict_group"] = "High-impact conflict"
matched_non_df["conflict_group"] = "Matched non-high-conflict"
coding_df = pd.concat([high_df, matched_non_df], axis=0, ignore_index=False).copy()

# Use de-identified IDs in coding and manuscript-facing exports.
coding_df = coding_df.reset_index(drop=False).rename(columns={"index": "source_index"})
coding_df["file_id"] = [f"P2A_{i + 1:04d}" for i in range(len(coding_df))]

print("Eligible files with text paths:", len(eligible))
print("High-impact conflict files selected:", len(high_df))
print("Matched non-high-conflict files selected:", len(matched_non_df))
print("Total files selected for OpenAI coding:", len(coding_df))
display(pd.DataFrame(match_notes))

# OUTPUT_DIR must be configured outside this cell. Keep identifiers and paths in the internal audit file only.
internal_cols = [
    "file_id",
    "source_index",
    "file_name",
    "text_path",
    "conflict_group",
    "language",
    "assignment_type",
    "major",
    "university",
    "year_grouping",
    "word_count",
    "combined_ai_rate_100",
    "detector_score_range",
]
internal_mapping = coding_df[
    [column for column in internal_cols if column in coding_df.columns]
].copy()
internal_mapping_path = (
    OUTPUT_DIR / "INTERNAL_file_id_text_path_mapping_not_for_manuscript.csv"
)
internal_mapping.to_csv(internal_mapping_path, index=False, encoding="utf-8-sig")
print("Saved internal mapping:", internal_mapping_path)

public_cols = [
    "file_id",
    "conflict_group",
    "language",
    "assignment_type",
    "major",
    "university",
    "year_grouping",
    "word_count",
    "combined_ai_rate_100",
    "detector_score_range",
    "detector_score_variance",
]
high_list_path = OUTPUT_DIR / "high_conflict_file_list.csv"
matched_list_path = OUTPUT_DIR / "matched_non_conflict_file_list.csv"

coding_df[coding_df["conflict_group"].eq("High-impact conflict")][
    [column for column in public_cols if column in coding_df.columns]
].to_csv(high_list_path, index=False, encoding="utf-8-sig")

coding_df[coding_df["conflict_group"].eq("Matched non-high-conflict")][
    [column for column in public_cols if column in coding_df.columns]
].to_csv(matched_list_path, index=False, encoding="utf-8-sig")

print("Saved:", high_list_path)
print("Saved:", matched_list_path)

## 6. Build safe de-identified excerpts

The trimmed files should already be de-identified. This section adds another scrubber before API use and sends only the excerpt, not file names or file paths.

In [ ]:
# =========================================================
# 6. Read text, scrub possible identifiers, and build excerpts
# =========================================================
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+\b", flags=re.IGNORECASE)
PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)")
ID_RE = re.compile(
    r"\b(?:ID|Id|id|Student\s*ID|University\s*ID|رقم|الرقم)\s*[:#\-]?\s*[A-Za-z0-9\-_/]{4,}\b"
)
NAME_LABEL_RE = re.compile(
    r"(?im)^\s*(?:name|student\s*name|instructor|teacher|doctor|professor|اسم|الاسم|الدكتور|د\.)\s*[:：].*$"
)
LONG_NUMBER_RE = re.compile(r"\b\d{7,}\b")

REPLACEMENTS = [
    (EMAIL_RE, "[EMAIL_REMOVED]"),
    (URL_RE, "[URL_REMOVED]"),
    (PHONE_RE, "[PHONE_REMOVED]"),
    (ID_RE, "[ID_REMOVED]"),
    (NAME_LABEL_RE, "[NAME_LINE_REMOVED]"),
    (LONG_NUMBER_RE, "[LONG_NUMBER_REMOVED]"),
]


def read_text_file(path):
    for enc in ["utf-8", "utf-8-sig", "cp1256", "latin-1"]:
        try:
            return Path(path).read_text(encoding=enc, errors="replace")
        except Exception:
            continue
    return ""


def safety_scrub_text(text):
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    out = text
    for pat, repl in REPLACEMENTS:
        out = pat.sub(repl, out)
    out = re.sub(r"[ \t]+", " ", out)
    out = re.sub(r"\n{3,}", "\n\n", out)
    return out.strip()


def word_tokens(text):
    return re.findall(r"\S+", text or "")


def build_safe_excerpt(text, chunk_words=700):
    scrubbed = safety_scrub_text(text)
    words = word_tokens(scrubbed)
    n = len(words)
    if n <= chunk_words * 2:
        excerpt = " ".join(words)
    else:
        first = words[:chunk_words]
        mid_start = max(0, n // 2 - chunk_words // 2)
        middle = words[mid_start : mid_start + chunk_words]
        last = words[-chunk_words:]
        excerpt = (
            "[BEGINNING EXCERPT]\n"
            + " ".join(first)
            + "\n\n[MIDDLE EXCERPT]\n"
            + " ".join(middle)
            + "\n\n[ENDING EXCERPT]\n"
            + " ".join(last)
        )
    return excerpt.strip(), n


texts = []
excerpt_lengths = []
for _, row in coding_df.iterrows():
    txt = read_text_file(row["text_path"])
    excerpt, n_words = build_safe_excerpt(txt, chunk_words=CHUNK_WORDS)
    texts.append(excerpt)
    excerpt_lengths.append(len(word_tokens(excerpt)))

coding_df["safe_excerpt"] = texts
coding_df["safe_excerpt_word_count"] = excerpt_lengths
coding_df["empty_excerpt"] = coding_df["safe_excerpt"].str.len().fillna(0).eq(0)

if coding_df["empty_excerpt"].any():
    empty_path = OUTPUT_DIR / "empty_excerpts_check.csv"
    coding_df[coding_df["empty_excerpt"]][
        [
            "file_id",
            "conflict_group",
            "language",
            "assignment_type",
            "word_count",
            "text_path",
        ]
    ].to_csv(empty_path, index=False, encoding="utf-8-sig")
    print("WARNING: Some excerpts are empty. See:", empty_path)

coding_ready_path = OUTPUT_DIR / "coding_ready_deidentified_excerpts.csv"
coding_df[
    [
        "file_id",
        "conflict_group",
        "language",
        "assignment_type",
        "major",
        "university",
        "year_grouping",
        "word_count",
        "combined_ai_rate_100",
        "detector_score_range",
        "safe_excerpt_word_count",
        "safe_excerpt",
    ]
].to_csv(coding_ready_path, index=False, encoding="utf-8-sig")
print("Saved coding-ready excerpts:", coding_ready_path)
display(
    coding_df[
        [
            "file_id",
            "conflict_group",
            "language",
            "assignment_type",
            "word_count",
            "safe_excerpt_word_count",
        ]
    ].head()
)

## 7. OpenAI structured-output feature coding

This cell uses OpenAI structured outputs with a strict JSON schema. It caches every coded file immediately so interrupted runs can resume. The API receives only `file_id`, metadata, and the safe excerpt.

In [ ]:
# =========================================================
# 7. OpenAI structured-output coding
# =========================================================
import os

FEATURE_NAMES_BINARY = [
    "template_artifacts",
    "formatting_artifacts",
    "reference_heavy_text",
    "formulaic_structure",
    "mixed_writing_style",
    "non_native_english_signals",
    "paraphrase_like_phrasing",
    "generic_or_overpolished_style",
    "abrupt_style_shifts",
    "short_or_fragmented_text",
    "long_multi_section_text",
]

feature_obj_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "present": {
            "type": "boolean",
            "description": "Whether the feature is observable in the excerpt.",
        },
        "confidence": {
            "type": "number",
            "minimum": 0,
            "maximum": 1,
            "description": "Confidence from 0 to 1.",
        },
        "evidence": {
            "type": "string",
            "description": "A short, non-identifying phrase or description supporting the code.",
        },
    },
    "required": ["present", "confidence", "evidence"],
}

FEATURE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "file_id": {"type": "string"},
        "language_detected": {
            "type": "string",
            "enum": ["Arabic", "English", "Mixed", "Code", "Unclear"],
        },
        "text_type": {"type": "string", "enum": ["prose", "code", "mixed", "unclear"]},
    },
    "required": ["file_id", "language_detected", "text_type"],
}
for fname in FEATURE_NAMES_BINARY:
    FEATURE_SCHEMA["properties"][fname] = feature_obj_schema
    FEATURE_SCHEMA["required"].append(fname)

FEATURE_SCHEMA["properties"]["technical_jargon_density"] = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "level": {"type": "string", "enum": ["low", "moderate", "high", "unclear"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "evidence": {"type": "string"},
    },
    "required": ["level", "confidence", "evidence"],
}
FEATURE_SCHEMA["required"].append("technical_jargon_density")
FEATURE_SCHEMA["properties"]["possible_reason_for_detector_disagreement"] = {
    "type": "string"
}
FEATURE_SCHEMA["properties"]["do_not_use_for_individual_judgment_note"] = {
    "type": "string"
}
FEATURE_SCHEMA["required"].extend(
    [
        "possible_reason_for_detector_disagreement",
        "do_not_use_for_individual_judgment_note",
    ]
)

RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "paper2_assignment_textual_feature_code",
        "strict": True,
        "schema": FEATURE_SCHEMA,
    },
}

SYSTEM_MESSAGE = (
    "You are coding de-identified academic assignment excerpts for a research study on why AI-writing detectors disagree. "
    "Do not decide whether the student used AI. Do not accuse any file of being AI-generated. "
    "Code only observable textual and formatting features that might plausibly affect detector scores. "
    "Return strict JSON only."
)

JUDGMENT_NOTE = "Detector disagreement and textual features are corpus-level indicators only and cannot determine whether this file was AI-generated."

CACHE_JSONL = CACHE_DIR / "openai_feature_codes_primary_assignment_corpus.jsonl"
CODES_CSV = OUTPUT_DIR / "openai_textual_feature_codes.csv"
ERRORS_CSV = OUTPUT_DIR / "openai_textual_feature_coding_errors.csv"

DATASET_SOURCE_TAG = "Paper2_primary_assignment_corpus_543"


def get_openai_key():
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    if IN_COLAB:
        try:
            from google.colab import userdata

            key = userdata.get("OPENAI_API_KEY")
            if key:
                return key
        except Exception:
            pass
    return None


def load_cache():
    rows = []
    if CACHE_JSONL.exists():
        for line in CACHE_JSONL.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                if obj.get("dataset_source") == DATASET_SOURCE_TAG:
                    rows.append(obj)
            except Exception:
                pass
    return {r.get("file_id"): r for r in rows if r.get("file_id")}


def append_cache(obj):
    obj = dict(obj)
    obj["dataset_source"] = DATASET_SOURCE_TAG
    with CACHE_JSONL.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def build_user_message(row):
    return (
        "Analyze the following de-identified assignment excerpt and code the requested textual features. "
        "Do not infer identity, authorship, or misconduct. "
        f"File ID = {row['file_id']}. "
        f"File metadata: language = {row.get('language', 'Unclear')}, "
        f"assignment type = {row.get('assignment_type', 'Unclear')}, "
        f"word count = {row.get('word_count', np.nan)}, "
        f"detector range = {row.get('detector_score_range', np.nan)}, "
        f"combined AI rate = {row.get('combined_ai_rate_100', np.nan)}. "
        "Text excerpt:\n"
        f"{row.get('safe_excerpt', '')}\n\n"
        f"Required caution note: {JUDGMENT_NOTE}"
    )


def validate_feature_json(obj, expected_file_id):
    if not isinstance(obj, dict):
        raise ValueError("Response is not a JSON object.")
    for key in FEATURE_SCHEMA["required"]:
        if key not in obj:
            raise ValueError(f"Missing required key: {key}")
    if str(obj.get("file_id")) != str(expected_file_id):
        # Correct file_id locally rather than failing; model sometimes echoes a slightly different value.
        obj["file_id"] = str(expected_file_id)
    return obj


def call_openai_structured(row, client, max_retries=4):
    user_msg = build_user_message(row)
    last_err = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": user_msg},
                ],
                response_format=RESPONSE_FORMAT,
                temperature=0,
            )
            content = response.choices[0].message.content
            obj = json.loads(content)
            obj = validate_feature_json(obj, row["file_id"])
            obj["openai_model"] = OPENAI_MODEL
            obj["coded_at_unix"] = time.time()
            return obj
        except Exception as e:
            last_err = e
            wait = min(60, 2**attempt + random.random())
            print(
                f"OpenAI coding error for {row['file_id']} on attempt {attempt+1}/{max_retries}: {e}. Waiting {wait:.1f}s"
            )
            time.sleep(wait)
    raise last_err


def flatten_code_record(obj):
    row = {
        "file_id": obj.get("file_id"),
        "language_detected": obj.get("language_detected"),
        "text_type": obj.get("text_type"),
        "possible_reason_for_detector_disagreement": obj.get(
            "possible_reason_for_detector_disagreement"
        ),
        "do_not_use_for_individual_judgment_note": obj.get(
            "do_not_use_for_individual_judgment_note"
        ),
        "openai_model": obj.get("openai_model"),
        "coded_at_unix": obj.get("coded_at_unix"),
    }
    for f in FEATURE_NAMES_BINARY:
        sub = obj.get(f, {}) or {}
        row[f"{f}_present"] = sub.get("present")
        row[f"{f}_confidence"] = sub.get("confidence")
        row[f"{f}_evidence"] = sub.get("evidence")
    tj = obj.get("technical_jargon_density", {}) or {}
    row["technical_jargon_density_level"] = tj.get("level")
    row["technical_jargon_density_confidence"] = tj.get("confidence")
    row["technical_jargon_density_evidence"] = tj.get("evidence")
    return row


cache = load_cache()
print("Cached primary-corpus feature codes:", len(cache))

errors = []
if RUN_OPENAI:
    api_key = get_openai_key()
    if not api_key:
        raise RuntimeError(
            "RUN_OPENAI=True but no OPENAI_API_KEY was found. In Colab, add it to Secrets as OPENAI_API_KEY, "
            "or set os.environ['OPENAI_API_KEY'] before running this cell."
        )
    try:
        from openai import OpenAI
    except Exception:
        raise ImportError(
            "The openai package is not installed. Set INSTALL_OPENAI_PACKAGE=True and rerun setup, or run: !pip install openai"
        )
    client = OpenAI(api_key=api_key)

    rows_to_code = coding_df[~coding_df["empty_excerpt"]].copy()
    if MAX_OPENAI_FILES is not None:
        rows_to_code = rows_to_code.head(int(MAX_OPENAI_FILES)).copy()

    print("Rows scheduled for OpenAI coding:", len(rows_to_code))
    for i, row in rows_to_code.iterrows():
        fid = row["file_id"]
        if fid in cache:
            continue
        try:
            obj = call_openai_structured(row, client)
            append_cache(obj)
            cache[fid] = obj
            if len(cache) % 10 == 0:
                print(f"Coded {len(cache)} files so far. Latest: {fid}")
        except Exception as e:
            err = {"file_id": fid, "error": str(e), "time": time.time()}
            errors.append(err)
            pd.DataFrame(errors).to_csv(ERRORS_CSV, index=False, encoding="utf-8-sig")
            print("FAILED:", fid, e)
else:
    print(
        "RUN_OPENAI=False. No API calls were made. Coding-ready excerpts and manual templates will still be exported."
    )

# Save current flattened codes.
flat_rows = [flatten_code_record(v) for v in cache.values()]
openai_codes = pd.DataFrame(flat_rows)
if len(openai_codes):
    # Merge manuscript-safe metadata, not file paths or file names.
    meta_cols = [
        "file_id",
        "conflict_group",
        "language",
        "assignment_type",
        "major",
        "university",
        "year_grouping",
        "word_count",
        "combined_ai_rate_100",
        "detector_score_range",
        "detector_score_variance",
    ]
    meta = coding_df[[c for c in meta_cols if c in coding_df.columns]].copy()
    openai_codes = meta.merge(openai_codes, on="file_id", how="left")
else:
    openai_codes = coding_df[
        [
            "file_id",
            "conflict_group",
            "language",
            "assignment_type",
            "major",
            "university",
            "year_grouping",
            "word_count",
            "combined_ai_rate_100",
            "detector_score_range",
            "detector_score_variance",
        ]
    ].copy()

openai_codes.to_csv(CODES_CSV, index=False, encoding="utf-8-sig")
print("Saved OpenAI feature codes:", CODES_CSV)
print(
    "Number of files with completed OpenAI codes:",
    (
        int(openai_codes["language_detected"].notna().sum())
        if "language_detected" in openai_codes.columns
        else 0
    ),
)
if errors:
    print("Errors saved:", ERRORS_CSV)
display(openai_codes.head())

## 8. Manual validation sample

This exports 20 files where possible: 10 high-impact conflict and 10 matched non-high-conflict. It includes coded features and blank human-review columns. It does not include file names or paths.

In [ ]:
# =========================================================
# 8. Manual validation export and optional agreement calculation
# =========================================================
def sample_by_group(data, group_label, n, seed):
    g = data[data["conflict_group"].eq(group_label)].copy()
    if len(g) == 0:
        return g
    return g.sample(n=min(n, len(g)), random_state=seed)


# Merge excerpts for validation, without file paths or names.
val_base = coding_df[["file_id", "safe_excerpt", "safe_excerpt_word_count"]].copy()
validation_pool = openai_codes.merge(val_base, on="file_id", how="left")

val_sample = pd.concat(
    [
        sample_by_group(validation_pool, "High-impact conflict", 10, RANDOM_SEED),
        sample_by_group(
            validation_pool, "Matched non-high-conflict", 10, RANDOM_SEED + 1
        ),
    ],
    ignore_index=True,
)

# Add blank human-review columns for binary features and notes.
for f in FEATURE_NAMES_BINARY:
    val_sample[f"human_{f}_present"] = ""
val_sample["human_technical_jargon_density_level"] = ""
val_sample["human_notes"] = ""
val_sample["human_no_ai_authorship_claim_confirmed"] = ""

manual_path = OUTPUT_DIR / "manual_validation_sample.xlsx"
with pd.ExcelWriter(manual_path, engine="openpyxl") as writer:
    val_sample.to_excel(writer, index=False, sheet_name="manual_validation")
print("Saved manual validation file:", manual_path)
display(
    val_sample[
        [
            "file_id",
            "conflict_group",
            "language",
            "assignment_type",
            "word_count",
            "safe_excerpt_word_count",
        ]
    ].head()
)

# Optional: after filling the file, save as manual_validation_sample_completed.xlsx and rerun this cell.
completed_path = OUTPUT_DIR / "manual_validation_sample_completed.xlsx"
validation_summary = pd.DataFrame()
if completed_path.exists():
    completed = pd.read_excel(completed_path)
    rows = []
    for f in FEATURE_NAMES_BINARY:
        ai_col = f"{f}_present"
        human_col = f"human_{f}_present"
        if ai_col not in completed.columns or human_col not in completed.columns:
            continue
        tmp = completed[[ai_col, human_col]].dropna().copy()
        tmp = tmp[tmp[human_col].astype(str).str.strip().ne("")]
        if tmp.empty:
            continue
        ai = tmp[ai_col].astype(str).str.lower().isin(["true", "1", "yes", "y"])
        hu = tmp[human_col].astype(str).str.lower().isin(["true", "1", "yes", "y"])
        agree = (ai == hu).mean()
        # Cohen kappa for binary.
        p_yes_ai = ai.mean()
        p_yes_hu = hu.mean()
        pe = p_yes_ai * p_yes_hu + (1 - p_yes_ai) * (1 - p_yes_hu)
        kappa = (agree - pe) / (1 - pe) if (1 - pe) else np.nan
        rows.append(
            {
                "feature": f,
                "n_reviewed": len(tmp),
                "percent_agreement": 100 * agree,
                "cohen_kappa": kappa,
            }
        )
    validation_summary = pd.DataFrame(rows)
    validation_summary.to_csv(
        OUTPUT_DIR / "manual_validation_summary.csv", index=False, encoding="utf-8-sig"
    )
    display(validation_summary)
else:
    print(
        "No completed manual validation file found yet. After review, save it as:",
        completed_path,
    )

## 9. Compare groups: numeric characteristics, detector patterns, and OpenAI-coded features

The comparisons are exploratory. Use Benjamini-Hochberg correction where multiple tests are run. Do not overinterpret small cells.

In [ ]:
def cliffs_delta(x, y):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    y = np.asarray(pd.Series(y).dropna(), dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = sum((xi > y).sum() for xi in x)
    lt = sum((xi < y).sum() for xi in x)
    return (gt - lt) / (len(x) * len(y))


def cramer_v_from_table(table):
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(table)
    n = table.values.sum()
    r, k = table.shape
    v = math.sqrt((chi2 / n) / max(1, min(k - 1, r - 1))) if n > 0 else np.nan
    return v, p


def fisher_or_chi2_binary(ct):
    if ct.shape == (2, 2):
        try:
            if (ct.values < 5).any():
                _, p = fisher_exact(ct.values)
                return "Fisher exact", p
        except Exception:
            pass
    try:
        _, p, _, _ = chi2_contingency(ct)
        return "Chi-square", p
    except Exception:
        return "Not tested", np.nan


def add_bh(df_in, p_col="p"):
    out = df_in.copy()
    if len(out) and p_col in out.columns:
        pvals = pd.to_numeric(out[p_col], errors="coerce")
        mask = pvals.notna()
        out["p_BH"] = np.nan
        if mask.sum() > 0:
            if STATSMODELS_AVAILABLE:
                out.loc[mask, "p_BH"] = multipletests(pvals[mask], method="fdr_bh")[1]
            else:
                # Benjamini-Hochberg fallback when statsmodels is unavailable.
                p = pvals[mask].values
                order = np.argsort(p)
                q = np.minimum.accumulate(
                    (p[order] * len(p) / np.arange(1, len(p) + 1))[::-1]
                )[::-1]
                out.loc[mask, "p_BH"] = q[np.argsort(order)]
    return out


comparison_groups = ["High-impact conflict", "Matched non-high-conflict"]

# Restrict all comparisons to the prespecified matched groups.
selected_numeric = coding_df.loc[
    coding_df["conflict_group"].isin(comparison_groups)
].copy()

compare_df = openai_codes.copy()
if "language_detected" in compare_df.columns:
    completed_mask = compare_df["language_detected"].notna()
else:
    completed_mask = pd.Series(False, index=compare_df.index)

completed_compare_df = compare_df.loc[
    completed_mask & compare_df["conflict_group"].isin(comparison_groups)
].copy()
print(
    "OpenAI-coded files available for feature comparisons:", len(completed_compare_df)
)

num_vars = [
    "word_count",
    "combined_ai_rate_100",
    "detector_score_range",
    "detector_score_variance",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]
num_rows = []

for var in num_vars:
    if var not in selected_numeric.columns:
        continue

    x = selected_numeric.loc[
        selected_numeric["conflict_group"].eq("High-impact conflict"), var
    ].dropna()
    y = selected_numeric.loc[
        selected_numeric["conflict_group"].eq("Matched non-high-conflict"), var
    ].dropna()
    if len(x) == 0 or len(y) == 0:
        continue

    try:
        _, p = mannwhitneyu(x, y, alternative="two-sided")
    except Exception:
        p = np.nan

    num_rows.append(
        {
            "variable": var,
            "high_conflict_n": len(x),
            "high_conflict_mean": x.mean(),
            "high_conflict_sd": x.std(),
            "high_conflict_median": x.median(),
            "high_conflict_IQR": x.quantile(0.75) - x.quantile(0.25),
            "matched_non_conflict_n": len(y),
            "matched_non_conflict_mean": y.mean(),
            "matched_non_conflict_sd": y.std(),
            "matched_non_conflict_median": y.median(),
            "matched_non_conflict_IQR": y.quantile(0.75) - y.quantile(0.25),
            "mann_whitney_p": p,
            "cliffs_delta": cliffs_delta(x, y),
        }
    )

num_tests = (
    add_bh(pd.DataFrame(num_rows), "mann_whitney_p") if num_rows else pd.DataFrame()
)
num_tests.to_csv(
    TABLE_DIR / "numeric_characteristics_by_conflict_status.csv",
    index=False,
    encoding="utf-8-sig",
)
display(num_tests)

cat_rows = []
for var in ["language", "assignment_type", "major", "university", "year_grouping"]:
    if var not in selected_numeric.columns:
        continue
    ct = pd.crosstab(
        selected_numeric[var].fillna("Missing"),
        selected_numeric["conflict_group"],
    )
    v, p = cramer_v_from_table(ct)
    cat_rows.append(
        {
            "variable": var,
            "cramers_v": v,
            "chi_square_p": p,
            "n_categories": ct.shape[0],
        }
    )

cat_tests = (
    add_bh(pd.DataFrame(cat_rows), "chi_square_p") if cat_rows else pd.DataFrame()
)
cat_tests.to_csv(
    TABLE_DIR / "categorical_characteristics_tests_by_conflict_status.csv",
    index=False,
    encoding="utf-8-sig",
)
display(cat_tests)

feature_rows = []
if len(completed_compare_df):
    for feature in FEATURE_NAMES_BINARY:
        col = f"{feature}_present"
        if col not in completed_compare_df.columns:
            continue

        tmp = completed_compare_df[["conflict_group", col]].dropna().copy()
        if tmp.empty:
            continue

        tmp[col] = tmp[col].astype(str).str.lower().isin(["true", "1", "yes", "y"])
        ct = pd.crosstab(tmp["conflict_group"], tmp[col]).reindex(
            index=comparison_groups,
            columns=[False, True],
            fill_value=0,
        )
        test_name, p = fisher_or_chi2_binary(ct)
        high_n = int(ct.loc["High-impact conflict"].sum())
        non_n = int(ct.loc["Matched non-high-conflict"].sum())
        high_pct = 100 * ct.loc["High-impact conflict", True] / max(1, high_n)
        non_pct = 100 * ct.loc["Matched non-high-conflict", True] / max(1, non_n)

        feature_rows.append(
            {
                "feature": feature,
                "high_conflict_n": high_n,
                "high_conflict_present_n": int(ct.loc["High-impact conflict", True]),
                "high_conflict_percent": high_pct,
                "matched_non_conflict_n": non_n,
                "matched_non_conflict_present_n": int(
                    ct.loc["Matched non-high-conflict", True]
                ),
                "matched_non_conflict_percent": non_pct,
                "difference_pp": high_pct - non_pct,
                "test": test_name,
                "p": p,
                "effect_size_name": "percentage-point difference",
                "effect_size": high_pct - non_pct,
            }
        )

    if "technical_jargon_density_level" in completed_compare_df.columns:
        tmp = (
            completed_compare_df[["conflict_group", "technical_jargon_density_level"]]
            .dropna()
            .copy()
        )
        if len(tmp):
            ct = pd.crosstab(
                tmp["technical_jargon_density_level"],
                tmp["conflict_group"],
            ).reindex(columns=comparison_groups, fill_value=0)
            v, p = cramer_v_from_table(ct)
            feature_rows.append(
                {
                    "feature": "technical_jargon_density_level",
                    "high_conflict_n": int(
                        (tmp["conflict_group"] == "High-impact conflict").sum()
                    ),
                    "high_conflict_present_n": np.nan,
                    "high_conflict_percent": np.nan,
                    "matched_non_conflict_n": int(
                        (tmp["conflict_group"] == "Matched non-high-conflict").sum()
                    ),
                    "matched_non_conflict_present_n": np.nan,
                    "matched_non_conflict_percent": np.nan,
                    "difference_pp": np.nan,
                    "test": "Chi-square",
                    "p": p,
                    "effect_size_name": "Cramer's V",
                    "effect_size": v,
                }
            )

feature_tests = (
    add_bh(pd.DataFrame(feature_rows), "p") if feature_rows else pd.DataFrame()
)
feature_path = OUTPUT_DIR / "feature_prevalence_by_conflict_status.csv"
feature_tests.to_csv(feature_path, index=False, encoding="utf-8-sig")
print("Saved feature prevalence:", feature_path)
display(feature_tests)

## 10. Charts and APA-ready tables

In [ ]:
def save_boxplot(data, y, filename, ylabel=None):
    groups = ["Matched non-high-conflict", "High-impact conflict"]
    values = [
        data.loc[data["conflict_group"].eq(group), y].dropna().values
        for group in groups
    ]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    try:
        ax.boxplot(values, tick_labels=groups, showmeans=True)
    except TypeError:  # Compatibility with older Matplotlib versions.
        ax.boxplot(values, labels=groups, showmeans=True)

    label = ylabel or y
    ax.set_ylabel(label)
    ax.set_title(f"{label} by conflict status")
    ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()

    output_path = FIG_DIR / filename
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    return output_path


chart_paths = []
if not coding_df.empty:
    chart_paths.append(
        save_boxplot(
            coding_df,
            "detector_score_range",
            "detector_range_by_conflict_status.png",
            "Detector score range",
        )
    )
    chart_paths.append(
        save_boxplot(
            coding_df, "word_count", "word_count_by_conflict_status.png", "Word count"
        )
    )

if not feature_tests.empty and "high_conflict_percent" in feature_tests.columns:
    plot_df = feature_tests.dropna(
        subset=["high_conflict_percent", "matched_non_conflict_percent"]
    ).copy()
    if not plot_df.empty:
        plot_df = plot_df.reindex(
            plot_df["difference_pp"].abs().sort_values(ascending=False).index
        ).head(10)
        x = np.arange(len(plot_df))
        width = 0.38

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(
            x - width / 2,
            plot_df["matched_non_conflict_percent"],
            width,
            label="Matched non-high-conflict",
        )
        ax.bar(
            x + width / 2,
            plot_df["high_conflict_percent"],
            width,
            label="High-impact conflict",
        )
        ax.set_ylabel("Feature prevalence (%)")
        ax.set_title("Top textual features by conflict status")
        ax.set_xticks(x)
        ax.set_xticklabels(
            plot_df["feature"].str.replace("_", " "), rotation=45, ha="right"
        )
        ax.legend()
        fig.tight_layout()

        output_path = FIG_DIR / "top_textual_features_by_conflict_status.png"
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
        plt.show()
        chart_paths.append(output_path)


characteristic_rows = []
for variable in ["language", "assignment_type", "major", "university", "year_grouping"]:
    if variable not in coding_df.columns:
        continue

    counts = pd.crosstab(
        coding_df[variable].fillna("Missing"), coding_df["conflict_group"]
    )
    for category, row in counts.iterrows():
        characteristic_rows.append(
            {
                "Variable": variable.replace("_", " ").title(),
                "Category": category,
                "High-conflict n": int(row.get("High-impact conflict", 0)),
                "Matched non-high-conflict n": int(
                    row.get("Matched non-high-conflict", 0)
                ),
            }
        )

for _, result in num_tests.iterrows():
    characteristic_rows.append(
        {
            "Variable": str(result["variable"]).replace("_", " ").title(),
            "Category": "Median (IQR)",
            "High-conflict n": (
                f"{result['high_conflict_median']:.2f} "
                f"({result['high_conflict_IQR']:.2f})"
            ),
            "Matched non-high-conflict n": (
                f"{result['matched_non_conflict_median']:.2f} "
                f"({result['matched_non_conflict_IQR']:.2f})"
            ),
        }
    )
table1 = pd.DataFrame(characteristic_rows)


detector_feature_cols = [
    column
    for column in [
        "gptzero_le_30",
        "gptzero_ge_70",
        "pangram_le_30",
        "pangram_ge_70",
        "sapling_le_30",
        "sapling_ge_70",
        "isgen_le_30",
        "isgen_ge_70",
    ]
    if column in coding_df.columns
]

detector_rows = []
for column in detector_feature_cols:
    data = coding_df[["conflict_group", column]].dropna().copy()
    if data.empty:
        continue

    counts = pd.crosstab(data["conflict_group"], data[column].astype(bool)).reindex(
        index=["High-impact conflict", "Matched non-high-conflict"],
        columns=[False, True],
        fill_value=0,
    )
    test_name, p_value = fisher_or_chi2_binary(counts)
    high_conflict_n = int(counts.loc["High-impact conflict"].sum())
    matched_non_conflict_n = int(counts.loc["Matched non-high-conflict"].sum())

    detector_rows.append(
        {
            "Pattern": column.replace("_", " "),
            "High-conflict present n": int(counts.loc["High-impact conflict", True]),
            "High-conflict %": (
                100 * counts.loc["High-impact conflict", True] / max(1, high_conflict_n)
            ),
            "Matched non-high-conflict present n": int(
                counts.loc["Matched non-high-conflict", True]
            ),
            "Matched non-high-conflict %": (
                100
                * counts.loc["Matched non-high-conflict", True]
                / max(1, matched_non_conflict_n)
            ),
            "Test": test_name,
            "p": p_value,
        }
    )

table2 = add_bh(pd.DataFrame(detector_rows), "p") if detector_rows else pd.DataFrame()

table3 = feature_tests.copy()
if not table3.empty:
    for column in [
        "high_conflict_percent",
        "matched_non_conflict_percent",
        "difference_pp",
        "p",
        "p_BH",
        "effect_size",
    ]:
        if column in table3.columns:
            table3[column] = pd.to_numeric(table3[column], errors="coerce").round(3)

table4 = (
    validation_summary.copy() if "validation_summary" in globals() else pd.DataFrame()
)

apa_xlsx = OUTPUT_DIR / "Paper2_OpenAI_Textual_Feature_APA_Tables.xlsx"
with pd.ExcelWriter(apa_xlsx, engine="openpyxl") as writer:
    table1.to_excel(writer, index=False, sheet_name="Table1_File_Characteristics")
    table2.to_excel(writer, index=False, sheet_name="Table2_Detector_Patterns")
    table3.to_excel(writer, index=False, sheet_name="Table3_Textual_Features")
    table4.to_excel(writer, index=False, sheet_name="Table4_Manual_Validation")
    num_tests.to_excel(writer, index=False, sheet_name="Numeric_Tests")
    cat_tests.to_excel(writer, index=False, sheet_name="Categorical_Tests")

print("Saved APA-ready Excel tables:", apa_xlsx)
print("Charts:")
for path in chart_paths:
    print(" -", path)

## 11. Manuscript-ready cautious interpretation

This paragraph is generated from the feature-prevalence table. Use it only after checking the results and manual validation.

In [ ]:
# =========================================================
# 11. Manuscript-ready interpretation
# =========================================================
def pretty_feature_name(f):
    return str(f).replace("_", " ")


def build_interpretation():
    n_high = int(df["high_impact_conflict"].sum())
    n_total = len(df)
    n_coded = (
        int(
            openai_codes.get("language_detected", pd.Series([], dtype=object))
            .notna()
            .sum()
        )
        if len(openai_codes)
        else 0
    )

    base = f"Using the predefined high-impact detector-conflict rule, {n_high} of {n_total} assignment files were flagged for substantial detector disagreement. "

    if n_coded == 0 or feature_tests is None or len(feature_tests) == 0:
        return base + (
            "The OpenAI-assisted textual feature coding was not completed in this run, so interpretation should remain limited to the numeric detector-disagreement patterns and the coding-ready file list. "
            "After de-identified excerpts are coded and manually checked, feature-prevalence results can be used to describe corpus-level textual patterns that may help explain why detectors disagreed. "
            "These results must not be interpreted as evidence that any individual file was AI-generated."
        )

    ft = feature_tests.dropna(
        subset=["high_conflict_percent", "matched_non_conflict_percent"]
    ).copy()
    if ft.empty:
        return base + (
            f"OpenAI-assisted textual feature coding was completed for {n_coded} files, but the feature table did not contain enough comparable binary features for interpretation. "
            "No individual file should be classified as AI-generated from these results."
        )

    sig = (
        ft[pd.to_numeric(ft.get("p_BH", np.nan), errors="coerce") < 0.05].copy()
        if "p_BH" in ft.columns
        else pd.DataFrame()
    )
    if len(sig):
        top = sig.sort_values("difference_pp", ascending=False).head(4)
        qualifier = "more common after Benjamini-Hochberg correction"
    else:
        top = ft.reindex(
            ft["difference_pp"].abs().sort_values(ascending=False).index
        ).head(4)
        qualifier = "descriptively more common, although not necessarily significant after correction"

    parts = []
    for _, r in top.iterrows():
        diff = float(r.get("difference_pp", np.nan))
        direction = "higher" if diff >= 0 else "lower"
        parts.append(
            f"{pretty_feature_name(r['feature'])} ({r['high_conflict_percent']:.1f}% vs. {r['matched_non_conflict_percent']:.1f}%; {direction} by {abs(diff):.1f} percentage points)"
        )
    feat_sentence = "; ".join(parts)

    return base + (
        f"In the exploratory OpenAI-assisted textual feature coding of de-identified excerpts, the features {qualifier} among high-conflict files were: {feat_sentence}. "
        "These patterns suggest that detector disagreement may be partly related to observable corpus-level textual or formatting characteristics, such as structure, style mixing, references, or genre-specific writing. "
        "However, these features are not ground truth indicators of AI authorship and should be interpreted only at the aggregate corpus level."
    )


interpretation = build_interpretation()
interpretation_path = OUTPUT_DIR / "manuscript_ready_interpretation.txt"
interpretation_path.write_text(interpretation, encoding="utf-8")
print(interpretation)
print("\nSaved manuscript-ready interpretation:", interpretation_path)

## 12. Final checklist

In [ ]:
# =========================================================
# 12. Final checklist
# =========================================================
checklist = pd.DataFrame(
    [
        {
            "item": "Primary Paper 2 assignment corpus loaded",
            "status": bool(len(df) == EXPECTED_N_FILES),
            "detail": f"Loaded {len(df)} rows",
        },
        {
            "item": "High-impact conflict flag matches expected 78 files",
            "status": bool(
                int(df["high_impact_conflict"].sum()) == EXPECTED_HIGH_CONFLICT_N
            ),
            "detail": f"Found {int(df['high_impact_conflict'].sum())}",
        },
        {
            "item": "Year-grouping issue fixed",
            "status": bool(
                not df["year_grouping"].astype(str).str.lower().eq("medical").any()
            ),
            "detail": str(df["year_grouping"].value_counts(dropna=False).to_dict()),
        },
        {
            "item": "Text paths loaded from primary trimmed assignments",
            "status": bool(
                df["text_path_exists"].sum() > 0
                and not df["text_path"]
                .astype(str)
                .str.contains("Testing Sample", case=False, na=False)
                .any()
            ),
            "detail": f"Resolved {int(df['text_path_exists'].sum())} paths",
        },
        {
            "item": "De-identification/scrubbing applied",
            "status": bool("safe_excerpt" in coding_df.columns),
            "detail": "Safety scrubber applied to excerpts",
        },
        {
            "item": "OpenAI coding completed or intentionally skipped",
            "status": True,
            "detail": f"RUN_OPENAI={RUN_OPENAI}; coded {int(openai_codes.get('language_detected', pd.Series([], dtype=object)).notna().sum()) if len(openai_codes) else 0} files",
        },
        {
            "item": "Manual validation file exported",
            "status": bool(manual_path.exists()),
            "detail": str(manual_path),
        },
        {
            "item": "APA tables exported",
            "status": bool(apa_xlsx.exists()),
            "detail": str(apa_xlsx),
        },
        {
            "item": "Manuscript paragraph exported",
            "status": bool(interpretation_path.exists()),
            "detail": str(interpretation_path),
        },
        {
            "item": "No individual AI-authorship claims made",
            "status": True,
            "detail": "Notebook language and outputs are corpus-level only",
        },
    ]
)
checklist_path = OUTPUT_DIR / "final_checklist.csv"
checklist.to_csv(checklist_path, index=False, encoding="utf-8-sig")
display(checklist)
print("Checklist saved:", checklist_path)